# primat documentation — figure generation

This notebook regenerates **every figure** embedded in the primat
documentation (`primat_documentation_v0.3.1.tex`).  Each cell is
self-contained and writes one PDF into `doc/figures/`.  Run *Kernel →
Restart & Run All* to refresh the whole figure set before recompiling the
LaTeX document.

The notebook only uses the public primat API (`primat.PRIMAT`, always running
the *Python* backend, plus the `plasma` / `network_data` building blocks), so
it doubles as a set of worked examples of how to pull the intermediate
ingredients (background temperatures, weak rates, reaction rates, abundance
histories) out of a run -- these are only reachable through the Python
backend (see the documentation's usage section, Sec. 1); the compiled C
engine that is the default for routine runs only returns the final result
dict (and, optionally, the time-evolution arrays).

In [1]:
# --- Setup -------------------------------------------------------------------
# Make the repo root importable whether the notebook is run from doc/ or root,
# select a non-interactive Matplotlib backend, and create the output folder.
import os
import sys

_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."
                        if os.path.basename(os.getcwd()) == "doc" else "."))
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

import numpy as np
import matplotlib
matplotlib.use("Agg")            # write files, never pop up a window
import matplotlib.pyplot as plt

from primat import PRIMAT
from primat.config import PRIMATConfig
from primat.network_data import (reaction_stoichiometry,
                                  compute_detailed_balance_coefficients)

FIGDIR = os.path.join(os.getcwd() if os.path.basename(os.getcwd()) == "doc"
                      else os.path.join(os.getcwd(), "doc"), "figures")
os.makedirs(FIGDIR, exist_ok=True)

plt.rcParams.update({
    "figure.dpi":      120,
    "font.size":       12,
    "axes.grid":       True,
    "grid.alpha":      0.25,
    "axes.labelsize":  13,
    "legend.fontsize": 10,
    "lines.linewidth": 1.8,
})

_CFG  = PRIMATConfig()
Q_MEV = (_CFG.mn - _CFG.mp)       # neutron-proton mass gap ~1.293 MeV
MeV_to_K = _CFG.MeV_to_Kelvin     # 1 MeV in kelvin
print(f"Q = m_n - m_p = {Q_MEV:.4f} MeV")


def savefig(fig, name):
    """Save *fig* as a tight PDF in FIGDIR and report the path."""
    path = os.path.join(FIGDIR, name)
    fig.savefig(path, bbox_inches="tight")
    print("wrote", os.path.relpath(path, _ROOT))
    plt.close(fig)

Q = m_n - m_p = 1.2933 MeV


In [2]:
# --- Reference run -----------------------------------------------------------
# A single standard-model run (small network, default Omega_b h^2) gives us the
# background history, the weak-rate interpolants and an abundance history that
# the figures below all reuse. Uses the Python backend (the default `PRIMAT`
# facade) so the intermediate background/nuclear attributes below are
# populated -- the compiled C engine only returns the final result dict.
base = PRIMAT({"network": "small"})
res  = base.solve()
print({k: float(v) if np.isscalar(v) else v for k, v in res.items()})

# Background temperature vectors (all in MeV).  Tg_vec runs high -> low; sort
# ascending once so np.interp can be used on it later.
Tg   = base.background.Tg_vec
order = np.argsort(Tg)
Tg_s    = Tg[order]
Tnue_s  = base.background.Tnue_vec[order]
Tnumu_s = base.background.Tnumu_vec[order]
Tnutau_s = base.background.Tnutau_vec[order]

# A few plasma helpers (rho_e, the QED pressure, the Hubble rate) are scalar
# functions, so wrap them for array input used by the plots below.
vrho_e  = np.vectorize(base.plasma.rho_e)
vspl    = np.vectorize(base.plasma.spl)
vPQED   = np.vectorize(base.plasma.PQEDofT)
vHubble = np.vectorize(base.background.Hubble)

[primat]  HT.

  MT.

  LT.

{'YPCMB': 0.24567248086944554, 'YPBBN': 0.24699878843093714, 'DoH': 2.4349969718810984e-05, 'He3oH': 1.0398016272573386e-05, 'He3oHe4': 0.00012678755831884724, 'Li7oH': 5.560169521521056e-10, 'Neff': 3.0439772985579183, 'Omeganurel': 5.6986373185272265, 'OneOverOmeganunr': 93.04303400064333}


  done.


## Section 2 — Plasma thermodynamics

In [3]:
# --- Fig: neutrino decoupling (flavour temperatures) -------------------------
ratio_asym = (4. / 11.) ** (1. / 3.)

fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.plot(Tg_s, Tnue_s / Tg_s,  label=r"$T_{\nu_e}/T_\gamma$")
ax.plot(Tg_s, Tnumu_s / Tg_s, label=r"$T_{\nu_\mu}/T_\gamma$", ls="--")
ax.axhline(ratio_asym, color="k", ls=":", lw=1.2,
           label=r"$(4/11)^{1/3}=%.4f$" % ratio_asym)
ax.set_xscale("log")
ax.set_xlabel(r"$T_\gamma$  [MeV]")
ax.set_ylabel(r"neutrino-to-photon temperature ratio")
ax.set_xlim(Tg_s.min(), 10.)
ax.invert_xaxis()
ax.legend(loc="lower left")
ax.set_title(r"Non-instantaneous neutrino decoupling — $N_{\rm eff}=%.5f$"
             % res["Neff"])
savefig(fig, "plasma_neutrino_decoupling.pdf")

wrote doc/figures/plasma_neutrino_decoupling.pdf


In [4]:
# --- Fig: effective relativistic degrees of freedom --------------------------
g_ref_rho = np.pi**2 / 30.       # energy density per bosonic dof / T^4
g_ref_s   = 2. * np.pi**2 / 45.  # entropy density per bosonic dof / T^3

rho_tot = (base.plasma.rho_g(Tg_s) + vrho_e(Tg_s)
           + base.plasma.rho_nu(Tnue_s) + base.plasma.rho_nu(Tnumu_s)
           + base.plasma.rho_nu(Tnutau_s))
s_nu = (4. / 3.) * (base.plasma.rho_nu(Tnue_s) / Tnue_s
                    + base.plasma.rho_nu(Tnumu_s) / Tnumu_s
                    + base.plasma.rho_nu(Tnutau_s) / Tnutau_s)
s_tot = vspl(Tg_s) + s_nu

g_rho = rho_tot / (g_ref_rho * Tg_s**4)
g_s   = s_tot   / (g_ref_s   * Tg_s**3)

fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.plot(Tg_s, g_rho, label=r"$g_{*\rho}$ (energy)")
ax.plot(Tg_s, g_s,   label=r"$g_{*s}$ (entropy)", ls="--")
ax.axhline(10.75, color="k", ls=":", lw=1.0)
ax.axvline(_CFG.me, color="tab:red", ls=":", lw=1.0,
           label=r"$T=m_e=%.3f$ MeV" % _CFG.me)
ax.set_xscale("log")
ax.set_xlabel(r"$T_\gamma$  [MeV]")
ax.set_ylabel(r"effective relativistic d.o.f.")
ax.set_xlim(1e-2, 10.)
ax.invert_xaxis()
ax.legend(loc="center left")
ax.set_title(r"Relativistic degrees of freedom through $e^+e^-$ annihilation")
savefig(fig, "plasma_gstar.pdf")

wrote doc/figures/plasma_gstar.pdf


In [5]:
# --- Fig: QED interaction-pressure correction --------------------------------
TQ = np.logspace(-2, 1, 400)                 # 0.01 -> 10 MeV
dP = vPQED(TQ)

fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.plot(TQ, -dP / TQ**4, color="tab:purple")
ax.axvline(_CFG.me, color="tab:red", ls=":", lw=1.0, label=r"$T=m_e$")
ax.set_xscale("log")
ax.set_xlabel(r"$T$  [MeV]")
ax.set_ylabel(r"$-\delta P_{\rm QED}/T^4$")
ax.legend()
ax.set_title(r"QED interaction-pressure correction (QED_corrections flag)")
savefig(fig, "plasma_qed_pressure.pdf")

wrote doc/figures/plasma_qed_pressure.pdf


## Section 3 — Weak interactions

In [6]:
# --- Fig: n<->p weak rates vs the Hubble rate --------------------------------
# The cached n<->p interpolants are tabulated against T_gamma in KELVIN.
NormWR = base.background.NormWeakRates
Tw  = np.logspace(np.log10(0.05), np.log10(3.0), 300)
TwK = Tw * MeV_to_K

Gnp = NormWR * base.background.weak_nTOp_frwrd(TwK)   # n -> p  [s^-1]
Gpn = NormWR * base.background.weak_nTOp_bkwrd(TwK)   # p -> n  [s^-1]

Tnue_w   = np.interp(Tw, Tg_s, Tnue_s)
Tnumu_w  = np.interp(Tw, Tg_s, Tnumu_s)
Tnutau_w = np.interp(Tw, Tg_s, Tnutau_s)
Hw = vHubble(Tw, Tnue_w, Tnumu_w, Tnutau_w)

# Highest-T crossing of Gamma_{n->p} and H = the n/p freeze-out.
sign_change = np.where(np.diff(np.sign(Gnp - Hw)) != 0)[0]
ifo = int(sign_change.max()) if sign_change.size else np.argmin(np.abs(Gnp - Hw))
T_fo = Tw[ifo]

fig, ax = plt.subplots(figsize=(6.4, 4.6))
ax.plot(Tw, Gnp, label=r"$\Gamma_{n\to p}$")
ax.plot(Tw, Gpn, label=r"$\Gamma_{p\to n}$", ls="--")
ax.plot(Tw, Hw,  label=r"Hubble rate $H$", color="k", ls=":")
ax.axvline(T_fo, color="tab:gray", lw=1.0)
ax.annotate(r"freeze-out $T\simeq%.2f$ MeV" % T_fo,
            xy=(T_fo, Hw[ifo]), xytext=(T_fo * 1.1, Hw[ifo] * 30),
            fontsize=9, color="tab:gray")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$T_\gamma$  [MeV]")
ax.set_ylabel(r"rate  [s$^{-1}$]")
ax.invert_xaxis()
ax.set_ylim(1e-3, 1e3)
ax.legend(loc="lower left")
ax.set_title(r"Weak rates vs expansion: the n/p freeze-out")
savefig(fig, "weak_rates_vs_hubble.pdf")

wrote doc/figures/weak_rates_vs_hubble.pdf


In [7]:
# --- Fig: decomposition of the corrections beyond Born -----------------------
# We isolate the three families of correction by toggling the relevant flags and
# comparing the forward rate Gamma_{n->p} to the crude Born rate.  Each run
# recomputes the rates (weak_rate_cache=False) so the toggles take effect.
def gnp_of(params):
    r = PRIMAT({"network": "small", "weak_rate_cache": False,
              "save_nTOp": False, **params})
    return r.background.NormWeakRates * r.background.weak_nTOp_frwrd(TwK)

# Born (crude) mode: all three additive corrections off (config.py's
# DEFAULT_PARAMS docstring: "Born (crude) mode = radiative_corrections=False,
# finite_mass_corrections=False, thermal_corrections=False").
g_born  = gnp_of({"radiative_corrections": False, "finite_mass_corrections": False,
                  "thermal_corrections": False, "spectral_distortions": False})
g_full  = gnp_of({})                                     # all corrections on (defaults)
# T=0 radiative + finite-mass + weak-magnetism only (no finite-T, no spectral):
g_rcfm  = gnp_of({"thermal_corrections": False, "spectral_distortions": False})
# add finite-temperature on top of RC+FM:
g_therm = gnp_of({"spectral_distortions": False})
# add spectral distortions on top of RC+FM:
g_spec  = gnp_of({"thermal_corrections": False})

fig, ax = plt.subplots(figsize=(6.8, 4.6))
ax.plot(Tw, 100 * (g_rcfm / g_born - 1),
        label=r"radiative + finite-mass + weak-magn. ($T=0$)")
ax.plot(Tw, 100 * (g_therm / g_rcfm - 1), ls="--",
        label=r"finite-temperature radiative")
ax.plot(Tw, 100 * (g_spec / g_rcfm - 1), ls="-.",
        label=r"neutrino spectral distortion")
ax.plot(Tw, 100 * (g_full / g_born - 1), color="k", lw=2.2,
        label=r"total ($\Gamma^{\rm full}/\Gamma^{\rm Born}-1$)")
ax.axhline(0.0, color="gray", lw=0.7)
ax.set_xscale("log")
ax.set_xlabel(r"$T_\gamma$  [MeV]")
ax.set_ylabel(r"correction to $\Gamma_{n\to p}$  [%]")
ax.invert_xaxis()
ax.legend(loc="best", fontsize=9)
ax.set_title(r"Corrections to the weak rate beyond the Born approximation")
savefig(fig, "weak_corrections_breakdown.pdf")

wrote doc/figures/weak_corrections_breakdown.pdf


In [8]:
# --- Fig: neutron mass fraction X_n through freeze-out and decay --------------
t_dom = base.nuclear.Y_of_t.x
tt = np.logspace(np.log10(max(t_dom.min(), 1e-2)),
                 np.log10(min(t_dom.max(), 1e4)), 400)
Tt = base.T_of_t(tt)                        # MeV
Xn = np.array([base.nuclear.Y_of_t(t)[0] for t in tt])   # index 0 = neutron
Xn_eq = 1.0 / (1.0 + np.exp(Q_MEV / Tt))

fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.plot(Tt, Xn,    label=r"$X_n$ (primat)")
ax.plot(Tt, Xn_eq, label=r"$X_n^{\rm eq}=1/(1+e^{Q/T})$", ls="--", color="k")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$T_\gamma$  [MeV]")
ax.set_ylabel(r"neutron mass fraction $X_n$")
ax.invert_xaxis()
ax.set_ylim(1e-3, 1.0)
ax.legend(loc="lower left")
ax.set_title(r"Neutron freeze-out and decay")
savefig(fig, "weak_neutron_fraction.pdf")

wrote doc/figures/weak_neutron_fraction.pdf


## Section 4 — Nuclear reactions

In [9]:
# --- Fig: forward and reverse thermonuclear rates ----------------------------
# The compiled forward-rate methods base.nucl.<rxn>_frwrd take T in KELVIN
# (internally T9 = T/1e9).  The reverse rate follows from detailed balance,
#   reverse(T9) = alpha * T9**beta * exp(gamma/T9) * forward(T9),
# with (alpha,beta,gamma) from compute_detailed_balance_coefficients.  Plotting
# both exposes where forward = reverse, i.e. where each capture is in
# equilibrium with its photo-dissociation (the deuterium bottleneck for n p->d).
T9 = np.logspace(np.log10(0.05), np.log10(5.0), 300)
reactions = [
    ("n_p__d_g",     r"$n\,p\rightleftharpoons d\,\gamma$"),
    ("d_p__He3_g",   r"$d\,p\rightleftharpoons {}^3{\rm He}\,\gamma$"),
    ("d_d__t_p",     r"$d\,d\rightleftharpoons t\,p$"),
    ("He3_a__Be7_g", r"${}^3{\rm He}\,\alpha\rightleftharpoons {}^7{\rm Be}\,\gamma$"),
]
colors = plt.cm.tab10(np.linspace(0, 1, 10))

fig, ax = plt.subplots(figsize=(7.0, 5.0))
for c, (rxn, label) in enumerate(reactions):
    reac, prod = reaction_stoichiometry(rxn)
    rl = [s for s, n in reac.items() for _ in range(n)]
    pl = [s for s, n in prod.items() for _ in range(n)]
    alpha, beta, gamma = compute_detailed_balance_coefficients(rl, pl, base.cfg)
    fwd_fn = getattr(base.nucl, rxn + "_frwrd")
    fwd = np.array([fwd_fn(t9 * 1e9) for t9 in T9])           # Kelvin argument
    rev = alpha * T9**beta * np.exp(gamma / T9) * fwd
    fwd = np.where(fwd > 0, fwd, np.nan)
    rev = np.where(rev > 0, rev, np.nan)
    ax.plot(T9, fwd, color=colors[c], label=label)
    ax.plot(T9, rev, color=colors[c], ls="--")
ax.plot([], [], color="gray", ls="-",  label="forward")
ax.plot([], [], color="gray", ls="--", label="reverse")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$T_9 = T/10^9\,$K")
ax.set_ylabel(r"$N_A\langle\sigma v\rangle$  [cm$^3$ s$^{-1}$ mol$^{-1}$]")
ax.set_ylim(1e-2, 1e12)
ax.legend(ncol=2, fontsize=8.5, loc="lower right")
ax.set_title(r"Forward (solid) and reverse (dashed) reaction rates")
savefig(fig, "nuclear_reaction_rates.pdf")

wrote doc/figures/nuclear_reaction_rates.pdf


In [10]:
# --- Fig: abundance evolution ------------------------------------------------
names = base.abundance_names
A_of  = base.A
ccol = plt.cm.tab10(np.linspace(0, 1, len(names)))
te = np.logspace(np.log10(max(t_dom.min(), 1e-1)),
                 np.log10(min(t_dom.max(), 1e5)), 400)
Yt = np.array([base.nuclear.Y_of_t(t) for t in te])

fig, ax = plt.subplots(figsize=(6.8, 5.0))
for j, nm in enumerate(names):
    ax.plot(te, A_of[nm] * Yt[:, j], color=ccol[j], label=nm)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$t$  [s]")
ax.set_ylabel(r"mass fraction $A_i Y_i$")
ax.set_ylim(1e-12, 2.0)
ax.legend(ncol=2, fontsize=9, loc="lower left")
ax.set_title(r"Abundance evolution (small network)")
savefig(fig, "nuclear_abundance_evolution.pdf")

wrote doc/figures/nuclear_abundance_evolution.pdf


In [11]:
# --- Fig: Schramm diagram with observational constraints ---------------------
# Final abundances vs the baryon density, the classic BBN prediction, scanned at
# the central nuclear rates.  We span the full eta_b = 1e-10 -> 1e-9 decade
# (the corresponding Omega_b h^2 follows from cfg.Omegabh2_to_eta0b).  Grey bands
# are the measured primordial values; the red band is the Planck baryon density.
# (Constraint values match notebooks/StandardPlots.ipynb.)
eta_per_ob2 = _CFG.Omegabh2_to_eta0b
eta_vec = np.logspace(-10, -9, 25)           # eta_b spans 1e-10 -> 1e-9
ob2_vec = eta_vec / eta_per_ob2              # corresponding Omega_b h^2

YP, DoH, He3oHe4, Li7oH = (np.zeros_like(eta_vec) for _ in range(4))
for i, ob2 in enumerate(ob2_vec):
    r = PRIMAT({"network": "small", "Omegabh2": ob2}).solve()
    YP[i]      = r["YPBBN"]
    DoH[i]     = r["DoH"]
    He3oHe4[i] = r["He3oH"] / (r["YPBBN"] / 4.)   # 3He/4He by number
    Li7oH[i]   = r["Li7oH"]

# Observational constraints (value, 1 sigma, reference, log-scale?)
obs = [
    (YP,      0.2458,   0.0013,   "Yeh+25",    r"$Y_P$",                 False),
    (DoH,     2.527e-5, 0.030e-5, "Cooke+18",  r"D/H",                   True),
    (He3oHe4, 1.15e-4,  0.24e-4,  "Cooke+26",  r"${}^3$He/${}^4$He",     True),
    (Li7oH,   1.58e-10, 0.31e-10, "Sbordone+10", r"${}^7$Li/H",         True),
]
eta_pl, eta_pl_e = 0.02285 * eta_per_ob2, 0.00016 * eta_per_ob2

fig, axs = plt.subplots(4, 1, figsize=(6.2, 11.0), sharex=True)
fig.subplots_adjust(hspace=0.06)
for ax, (y, oval, oerr, ref, lab, logy) in zip(axs, obs):
    ax.plot(eta_vec, y, color="tab:blue", lw=2)
    ax.axhspan(oval - oerr, oval + oerr, color="0.5", alpha=0.35,
               label="obs: %s" % ref)
    ax.axvspan(eta_pl - eta_pl_e, eta_pl + eta_pl_e, color="red", alpha=0.15)
    ax.set_ylabel(lab)
    ax.set_xscale("log")            # eta_b spans a full decade
    if logy:
        ax.set_yscale("log")
    ax.legend(loc="best", fontsize=9)
axs[-1].set_xlabel(r"$\eta_b = n_b/n_\gamma$")
axs[-1].set_xlim(1e-10, 1e-9)
axs[0].set_title(r"Schramm diagram with observational constraints"
                 "\n(grey: measured value; red: Planck $\\Omega_b h^2$)")
savefig(fig, "nuclear_schramm.pdf")

[primat]  HT.

  MT.

  LT.

  done.


[primat]  HT.

...

[primat]  HT.

  MT.

  LT.

  done.


wrote doc/figures/nuclear_schramm.pdf


## Section 5 — Sensitivity of abundances

In [12]:
# --- Fig: sensitivity heat-map -----------------------------------------------
# Logarithmic sensitivity S = d ln(O)/d ln(p) of each observable O to each
# parameter p, by a symmetric 1% finite difference.  Nuclear rates are varied
# through the additive delta_<rxn> knob (median * (1 + delta), see
# network_data.NetworkDefinition.apply_variations); physical parameters
# directly.  This is the same construction as notebooks/Sensitivity.ipynb.
FIDUCIAL = {"Omegabh2": 0.022425, "network": "small"}
OBS = ["YPBBN", "DoH", "He3oH", "Li7oH"]
OBS_LAB = [r"$Y_P$", r"D/H", r"$^3$He/H", r"$^7$Li/H"]
DELTA = 0.01


def run_obs(params):
    r = PRIMAT({**FIDUCIAL, **params}).primat_results()
    return np.array([r[q] for q in OBS])


def log_sens(plus, minus):
    return (np.log(plus) - np.log(minus)) / (2 * np.log(1 + DELTA))


# The small network's 12 reactions, in their canonical (small.txt) order, so
# RATE_NAMES and RATE_LAB line up entry-by-entry.
RATE_NAMES = ["n_p__d_g", "d_p__He3_g", "d_d__He3_n", "d_d__t_p", "t_p__a_g",
              "t_d__a_n", "t_a__Li7_g", "He3_n__t_p", "He3_d__a_p",
              "He3_a__Be7_g", "Be7_n__Li7_p", "Li7_p__a_a"]
RATE_LAB = [r"$np\to d\gamma$", r"$dp\to{}^3$He$\gamma$", r"$dd\to{}^3$He$n$",
            r"$dd\to tp$", r"$tp\to\alpha\gamma$", r"$td\to\alpha n$",
            r"$t\alpha\to{}^7$Li$\gamma$", r"${}^3$He$n\to tp$",
            r"${}^3$He$d\to\alpha p$", r"${}^3$He$\alpha\to{}^7$Be$\gamma$",
            r"${}^7$Be$n\to{}^7$Li$p$", r"${}^7$Li$p\to\alpha\alpha$"]

rate_sens = np.zeros((len(RATE_NAMES), len(OBS)))
for i, rxn in enumerate(RATE_NAMES):
    key = f"delta_{rxn}"
    rp = run_obs({key: +DELTA})
    rm = run_obs({key: -DELTA})
    rate_sens[i] = log_sens(rp, rm)

cfg0 = PRIMATConfig()
phys = {
    r"$\tau_n$":             lambda d: {"tau_n": cfg0.tau_n * (1 + d)},
    r"$G_N$":                lambda d: {"GN": cfg0.GN * (1 + d)},
    r"$\Omega_b h^2$":       lambda d: {"Omegabh2": FIDUCIAL["Omegabh2"] * (1 + d)},
    r"$\Delta N_{\rm eff}$": lambda d: {"DeltaNeff": d / DELTA},
}
phys_sens = np.zeros((len(phys), len(OBS)))
for i, (lab, vary) in enumerate(phys.items()):
    phys_sens[i] = log_sens(run_obs(vary(+DELTA)), run_obs(vary(-DELTA)))

all_lab  = RATE_LAB + list(phys.keys())
all_sens = np.vstack([rate_sens, phys_sens])

fig, ax = plt.subplots(figsize=(6.4, 0.42 * len(all_lab) + 1.6))
vmax = np.max(np.abs(all_sens))
im = ax.imshow(all_sens, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
for i in range(len(all_lab)):
    for j in range(len(OBS)):
        v = all_sens[i, j]
        ax.text(j, i, f"{v:+.2f}", ha="center", va="center", fontsize=7.5,
                color="white" if abs(v) > 0.5 * vmax else "black")
ax.axhline(len(RATE_LAB) - 0.5, color="black", lw=2)
ax.set_xticks(range(len(OBS)))
ax.set_xticklabels(OBS_LAB, fontsize=11)
ax.set_yticks(range(len(all_lab)))
ax.set_yticklabels(all_lab, fontsize=8.5)
ax.xaxis.set_ticks_position("top")
plt.colorbar(im, ax=ax, label=r"$\partial\ln O/\partial\ln p$", shrink=0.6)
ax.set_title("BBN sensitivity matrix\n(response per 1% change)",
             fontsize=10, pad=34)
savefig(fig, "sensitivity_heatmap.pdf")

print("\nAll figures written to", os.path.relpath(FIGDIR, _ROOT))

[primat]  HT.

  MT.

  LT.

  done.


[primat]  HT.

...

[primat]  HT.

  MT.

  LT.

  done.


wrote doc/figures/sensitivity_heatmap.pdf

All figures written to doc/figures
